

| Secret name | Value |
|---|---|
| `DB_HOST` | e.g. `ep-xxxx.us-east-2.aws.neon.tech` (from Neon/Supabase) |
| `DB_PORT` | usually `5432` |
| `DB_NAME` | your database name |
| `DB_USER` | your database user |
| `DB_PASSWORD` | your database password |
| `JWT_SECRET` | any long random string (generate one in the next cell) |
| `SMTP_EMAIL` | your Gmail address |
| `SMTP_APP_PASSWORD` | 16-character Gmail **App Password** (not your real password) |
| `NGROK_AUTHTOKEN` | from https://dashboard.ngrok.com/get-started/your-authtoken |


neondb_owner

In [1]:
!pip install -q streamlit==1.38.0 psycopg2-binary==2.9.9 PyJWT==2.9.0 bcrypt==4.2.0 \
    python-dotenv==1.0.1 email-validator==2.2.0 pyngrok==7.2.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.8/273.8 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.9/82.9 kB 5.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.3.0 requires tenacity<10,>=9, but you have tenacity 8.5.0 which is incompatible.
google-adk 2.3.0 requires watchdog<7,>=6, but you have watchdog 4.0.2 which is incompatible.


In [2]:
from google.colab import userdata

required_secrets = [
    "DB_HOST", "DB_PORT", "DB_NAME", "DB_USER", "DB_PASSWORD",
    "JWT_SECRET", "SMTP_EMAIL", "SMTP_APP_PASSWORD", "NGROK_AUTHTOKEN",
]

values = {}
missing = []
for key in required_secrets:
    try:
        values[key] = userdata.get(key)
    except Exception:
        missing.append(key)

if missing:
    raise RuntimeError(
        f"Missing Colab secrets: {missing}. "
        f"Add them via the key icon in the left sidebar, then re-run this cell."
    )

env_content = f'''DB_HOST={values["DB_HOST"]}
DB_PORT={values["DB_PORT"]}
DB_NAME={values["DB_NAME"]}
DB_USER={values["DB_USER"]}
DB_PASSWORD={values["DB_PASSWORD"]}

JWT_SECRET={values["JWT_SECRET"]}
JWT_ALGORITHM=HS256
JWT_EXPIRY_MINUTES=60

SMTP_HOST=smtp.gmail.com
SMTP_PORT=587
SMTP_EMAIL={values["SMTP_EMAIL"]}
SMTP_APP_PASSWORD={values["SMTP_APP_PASSWORD"]}

OTP_EXPIRY_MINUTES=10
'''

with open(".env", "w") as f:
    f.write(env_content)

print("Wrote .env with", len(values), "secrets loaded.")

Wrote .env with 9 secrets loaded.


In [3]:
%%writefile db.py
"""
db.py
PostgreSQL connection handling + schema initialization.
"""

import os
import psycopg2
from psycopg2.extras import RealDictCursor
from contextlib import contextmanager
from dotenv import load_dotenv

load_dotenv()

DB_CONFIG = {
    "host": os.getenv("DB_HOST", "localhost"),
    "port": os.getenv("DB_PORT", "5432"),
    "dbname": os.getenv("DB_NAME", "auth_app"),
    "user": os.getenv("DB_USER", "postgres"),
    "password": os.getenv("DB_PASSWORD", ""),
    "sslmode": "require",
}


@contextmanager
def get_connection():
    """Yield a PostgreSQL connection, always closed on exit."""
    conn = psycopg2.connect(**DB_CONFIG)
    try:
        yield conn
    finally:
        conn.close()


@contextmanager
def get_cursor(commit: bool = False):
    """Yield a RealDictCursor. Commits if commit=True and no exception occurred."""
    with get_connection() as conn:
        cur = conn.cursor(cursor_factory=RealDictCursor)
        try:
            yield cur
            if commit:
                conn.commit()
        except Exception:
            conn.rollback()
            raise
        finally:
            cur.close()


def init_db():
    """Create tables if they don't exist. Safe to call on every app startup."""
    with get_cursor(commit=True) as cur:
        cur.execute("""
            CREATE TABLE IF NOT EXISTS users (
                id SERIAL PRIMARY KEY,
                username VARCHAR(50) UNIQUE NOT NULL,
                email VARCHAR(255) UNIQUE NOT NULL,
                password_hash VARCHAR(255) NOT NULL,
                is_verified BOOLEAN NOT NULL DEFAULT TRUE,
                created_at TIMESTAMP NOT NULL DEFAULT NOW()
            );
        """)

        cur.execute("""
            CREATE TABLE IF NOT EXISTS otp_codes (
                id SERIAL PRIMARY KEY,
                email VARCHAR(255) NOT NULL,
                code VARCHAR(6) NOT NULL,
                purpose VARCHAR(20) NOT NULL,
                expires_at TIMESTAMP NOT NULL,
                used BOOLEAN NOT NULL DEFAULT FALSE,
                created_at TIMESTAMP NOT NULL DEFAULT NOW()
            );
        """)

        cur.execute("""
            CREATE INDEX IF NOT EXISTS idx_otp_email_purpose
            ON otp_codes (email, purpose);
        """)

        # Check and add 'name' column to users table if it does not exist
        cur.execute("ALTER TABLE users ADD COLUMN IF NOT EXISTS name VARCHAR(100);")

    print("Database initialized (tables ensured).")


if __name__ == "__main__":
    init_db()


Writing db.py


In [4]:
%%writefile auth.py
"""
auth.py
Password hashing, JWT issuing/verification, and user data-access functions.
"""

import os
import jwt
import bcrypt
import random
import string
from datetime import datetime, timedelta, timezone
from dotenv import load_dotenv

from db import get_cursor

load_dotenv()

JWT_SECRET = os.getenv("JWT_SECRET", "super-secret-default-key")
JWT_ALGORITHM = os.getenv("JWT_ALGORITHM", "HS256")
JWT_EXPIRY_MINUTES = int(os.getenv("JWT_EXPIRY_MINUTES", "60"))
OTP_EXPIRY_MINUTES = int(os.getenv("OTP_EXPIRY_MINUTES", "10"))


def hash_password(plain_password: str) -> str:
    return bcrypt.hashpw(plain_password.encode("utf-8"), bcrypt.gensalt()).decode("utf-8")


def verify_password(plain_password: str, password_hash: str) -> bool:
    return bcrypt.checkpw(plain_password.encode("utf-8"), password_hash.encode("utf-8"))


def create_jwt(user_id: int, username: str, email: str) -> str:
    payload = {
        "user_id": user_id,
        "username": username,
        "email": email,
        "exp": datetime.now(timezone.utc) + timedelta(minutes=JWT_EXPIRY_MINUTES),
        "iat": datetime.now(timezone.utc),
    }
    return jwt.encode(payload, JWT_SECRET, algorithm=JWT_ALGORITHM)


def decode_jwt(token: str):
    """Returns the payload dict if valid, else None."""
    try:
        return jwt.decode(token, JWT_SECRET, algorithms=[JWT_ALGORITHM])
    except jwt.ExpiredSignatureError:
        return None
    except jwt.InvalidTokenError:
        return None


def get_user_by_email(email: str):
    with get_cursor() as cur:
        cur.execute("SELECT * FROM users WHERE email = %s;", (email,))
        return cur.fetchone()


def get_user_by_username(username: str):
    with get_cursor() as cur:
        cur.execute("SELECT * FROM users WHERE username = %s;", (username,))
        return cur.fetchone()


def create_user(name: str, username: str, email: str, password: str):
    password_hash = hash_password(password)
    with get_cursor(commit=True) as cur:
        cur.execute(
            """
            INSERT INTO users (name, username, email, password_hash, is_verified)
            VALUES (%s, %s, %s, %s, TRUE)
            RETURNING id, name, username, email;
            """,
            (name, username, email, password_hash),
        )
        return cur.fetchone()


def mark_user_verified(email: str):
    with get_cursor(commit=True) as cur:
        cur.execute("UPDATE users SET is_verified = TRUE WHERE email = %s;", (email,))


def update_user_password(email: str, new_password: str):
    password_hash = hash_password(new_password)
    with get_cursor(commit=True) as cur:
        cur.execute(
            "UPDATE users SET password_hash = %s WHERE email = %s;",
            (password_hash, email),
        )


def generate_otp() -> str:
    return "".join(random.choices(string.digits, k=6))


def store_otp(email: str, code: str, purpose: str):
    """purpose: 'signup' or 'password_reset'"""
    expires_at = datetime.now(timezone.utc) + timedelta(minutes=OTP_EXPIRY_MINUTES)
    with get_cursor(commit=True) as cur:
        cur.execute(
            "UPDATE otp_codes SET used = TRUE WHERE email = %s AND purpose = %s AND used = FALSE;",
            (email, purpose),
        )
        cur.execute(
            """
            INSERT INTO otp_codes (email, code, purpose, expires_at)
            VALUES (%s, %s, %s, %s);
            """,
            (email, code, purpose, expires_at),
        )


def verify_otp(email: str, code: str, purpose: str) -> bool:
    with get_cursor(commit=True) as cur:
        cur.execute(
            """
            SELECT * FROM otp_codes
            WHERE email = %s AND purpose = %s AND used = FALSE
            ORDER BY created_at DESC LIMIT 1;
            """,
            (email, purpose),
        )
        row = cur.fetchone()
        if not row:
            return False

        expires_at = row["expires_at"]
        now = datetime.now(expires_at.tzinfo) if expires_at.tzinfo else datetime.now()

        if now > expires_at:
            return False
        if row["code"] != code:
            return False

        cur.execute("UPDATE otp_codes SET used = TRUE WHERE id = %s;", (row["id"],))
        return True


def get_latest_otp(email: str, purpose: str) -> str:
    """Fetch the last unexpired, unused OTP code for local debugging/demo mode."""
    with get_cursor() as cur:
        cur.execute(
            """
            SELECT code FROM otp_codes
            WHERE email = %s AND purpose = %s AND used = FALSE
            ORDER BY created_at DESC LIMIT 1;
            """,
            (email, purpose),
        )
        row = cur.fetchone()
        return row["code"] if row else None


Writing auth.py


In [5]:
%%writefile email_utils.py
"""
email_utils.py
Sends OTP emails using Gmail SMTP (smtp.gmail.com, port 587, STARTTLS).
Includes console fallback when credentials are not configured.
"""

import os
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from dotenv import load_dotenv

load_dotenv()

SMTP_HOST = os.getenv("SMTP_HOST", "smtp.gmail.com")
SMTP_PORT = int(os.getenv("SMTP_PORT", "587"))
SMTP_EMAIL = os.getenv("SMTP_EMAIL")
SMTP_APP_PASSWORD = os.getenv("SMTP_APP_PASSWORD")


def send_email(to_email: str, subject: str, body: str):
    # Check if SMTP is configured correctly and not a template value
    is_dummy = (
        not SMTP_EMAIL or
        not SMTP_APP_PASSWORD or
        SMTP_EMAIL.startswith("your") or
        SMTP_APP_PASSWORD.startswith("your") or
        "@" not in SMTP_EMAIL
    )

    if is_dummy:
        print(f"\n[DEMO EMAIL CLIENT] To: {to_email}\nSubject: {subject}\nBody:\n{body}\n")
        return True, "DEMO_MODE: Email printed to console successfully."

    msg = MIMEMultipart()
    msg["From"] = SMTP_EMAIL
    msg["To"] = to_email
    msg["Subject"] = subject
    msg.attach(MIMEText(body, "plain"))

    try:
        with smtplib.SMTP(SMTP_HOST, SMTP_PORT, timeout=15) as server:
            server.ehlo()
            server.starttls()
            server.ehlo()
            server.login(SMTP_EMAIL, SMTP_APP_PASSWORD)
            server.sendmail(SMTP_EMAIL, to_email, msg.as_string())
        return True, "Email sent."
    except smtplib.SMTPAuthenticationError:
        return False, "SMTP auth failed. Check SMTP_EMAIL / SMTP_APP_PASSWORD (use a Gmail App Password)."
    except Exception as e:
        return False, f"Failed to send email: {e}"


def send_otp_email(to_email: str, otp_code: str, purpose: str):
    if purpose == "signup":
        subject = "Your Verification Code"
        body = (
            f"Welcome!\n\nYour verification code is: {otp_code}\n\n"
            f"This code expires in a few minutes. If you did not request this, "
            f"you can safely ignore this email."
        )
    else:
        subject = "Your Password Reset Code"
        body = (
            f"We received a request to reset your password.\n\n"
            f"Your reset code is: {otp_code}\n\n"
            f"This code expires in a few minutes. If you did not request this, "
            f"you can safely ignore this email."
        )
    return send_email(to_email, subject, body)


Writing email_utils.py


In [18]:
!pip install python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 1.2 MB/s eta 0:00:00


In [23]:
%%writefile app.py
"""
app.py
Streamlit login/signup/password-reset app backed by PostgreSQL,
using JWT session tokens and Gmail SMTP for OTP delivery.
"""
import docx
import re
import time
import pandas as pd
import streamlit as st
import docx
from email_validator import validate_email, EmailNotValidError
import os

from db import init_db
from auth import (
    create_jwt,
    decode_jwt,
    get_user_by_email,
    get_user_by_username,
    create_user,
    mark_user_verified,
    update_user_password,
    verify_password,
    generate_otp,
    store_otp,
    verify_otp,
    get_latest_otp,
)
from email_utils import send_otp_email

# Set page configuration with a modern look
st.set_page_config(
    page_title="Reflect & Support",
    layout="centered",
    initial_sidebar_state="collapsed",
)

# Custom CSS for a fun, light-themed aesthetic
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Fredoka:wght@400;500;600;700&family=Outfit:wght@300;400;500;600;700&display=swap');

:root {
    --coral: #FF6B6B;
    --teal: #3FBFB0;
    --sunshine: #FFC93C;
    --grape: #9B8DF3;
    --cream: #FFFBF3;
    --ink: #2D2A32;
}

/* Playful gradient backdrop */
.stApp {
    background:
        radial-gradient(circle at 8% 15%, rgba(255,201,60,0.18) 0%, transparent 32%),
        radial-gradient(circle at 92% 12%, rgba(155,141,243,0.16) 0%, transparent 34%),
        radial-gradient(circle at 15% 92%, rgba(63,191,176,0.16) 0%, transparent 32%),
        radial-gradient(circle at 88% 88%, rgba(255,107,107,0.14) 0%, transparent 34%),
        var(--cream);
}

html, body, [class*="css"], .stMarkdown {
    font-family: 'Outfit', sans-serif;
    color: var(--ink);
}

/* Bouncy, colorful headline font */
.wellness-title {
    font-family: 'Fredoka', sans-serif;
    background: linear-gradient(90deg, var(--coral) 0%, var(--grape) 50%, var(--teal) 100%);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    font-size: 2.6rem;
    font-weight: 600;
    margin-bottom: 1rem;
    letter-spacing: 0.5px;
}

/* Card container with a playful colored edge */
.wellness-card {
    background-color: #ffffff;
    border: 1px solid #f1e9da;
    border-left: 5px solid var(--grape);
    border-radius: 16px;
    padding: 24px;
    margin-bottom: 20px;
    box-shadow: 0 6px 18px rgba(155, 141, 243, 0.12);
    color: var(--ink);
    transition: transform 0.2s ease, box-shadow 0.2s ease;
}

.wellness-card:hover {
    transform: translateY(-3px);
    box-shadow: 0 10px 22px rgba(155, 141, 243, 0.18);
}

/* Metric cards each get their own rotating accent color */
.wellness-metric-card {
    background: #ffffff;
    border: 1px solid #f1e9da;
    border-radius: 14px;
    padding: 16px;
    text-align: center;
    box-shadow: 0 4px 12px rgba(0, 0, 0, 0.05);
    color: var(--ink);
    position: relative;
    overflow: hidden;
}

.wellness-metric-card::before {
    content: "";
    position: absolute;
    top: 0; left: 0; right: 0;
    height: 4px;
    background: linear-gradient(90deg, var(--coral), var(--sunshine), var(--teal), var(--grape));
}

/* Buttons: rounded, springy hover, colorful border */
.stButton>button {
    border-radius: 999px;
    font-family: 'Fredoka', sans-serif;
    font-weight: 500;
    border: 2px solid var(--grape);
    transition: all 0.2s cubic-bezier(0.34, 1.56, 0.64, 1);
}

.stButton>button:hover {
    transform: translateY(-2px) scale(1.02);
    box-shadow: 0 6px 14px rgba(155, 141, 243, 0.25);
    background-color: #faf7ff;
    border-color: var(--coral);
    color: var(--coral);
}

.stButton>button[kind="primary"] {
    background: linear-gradient(90deg, var(--coral), var(--grape));
    border: none;
    color: white;
}

.stButton>button[kind="primary"]:hover {
    background: linear-gradient(90deg, var(--grape), var(--teal));
    color: white;
}

/* Playful rounded inputs */
.stTextInput>div>div>input {
    border-radius: 10px;
    border: 1.5px solid #e9e2d3;
}

.stTextInput>div>div>input:focus {
    border-color: var(--teal);
    box-shadow: 0 0 0 2px rgba(63, 191, 176, 0.18);
}
</style>
""", unsafe_allow_html=True)


@st.cache_resource
def _init_db_once():
    init_db()
    return True


_init_db_once()

# Initialize session state variables
defaults = {
    "page": "home",
    "jwt_token": None,
    "pending_email": None,
    "pending_username": None,
}
for key, val in defaults.items():
    if key not in st.session_state:
        st.session_state[key] = val


def goto(page: str):
    st.session_state.page = page
    st.rerun()


def is_valid_password(pw: str):
    if len(pw) < 8:
        return False, "Password must be at least 8 characters."
    if not re.search(r"[A-Za-z]", pw) or not re.search(r"[0-9]", pw):
        return False, "Password must contain both letters and numbers."
    return True, ""


def extract_text_from_upload(uploaded_file) -> str:
    """Pull plain text out of a csv, txt, or docx upload so it can be sent to the LLM."""
    filename = uploaded_file.name.lower()

    if filename.endswith(".csv"):
        df = pd.read_csv(uploaded_file)
        return df.to_string(index=False)

    if filename.endswith(".txt"):
        return uploaded_file.read().decode("utf-8", errors="ignore")

    if filename.endswith(".docx"):
        document = docx.Document(uploaded_file)
        return "\n".join(p.text for p in document.paragraphs if p.text.strip())

    return ""


def get_emotional_support(document_text: str) -> dict:
    """
    Placeholder for the LLM call.

    Wire this up to whichever model you choose. Send `document_text` along with
    a prompt asking for:
      - a short, warm supportive reflection (a few sentences)
      - a small list of concrete activities suited to what the person shared

    Must return a dict shaped exactly like this:
    {
        "support_message": str,
        "activities": [str, str, ...]
    }
    """
    return {
        "support_message": (
            "This is placeholder text. Connect this function to your LLM of choice "
            "so it can read what was uploaded and respond with a real, personalized "
            "reflection."
        ),
        "activities": [
            "Placeholder activity one",
            "Placeholder activity two",
            "Placeholder activity three",
        ],
    }

# Check for active JWT session
if st.session_state.jwt_token:
    payload = decode_jwt(st.session_state.jwt_token)
    if payload:
        # --- WELCOME / DOCUMENT UPLOAD PAGE (DASHBOARD) ---
        st.markdown(f"<h1 class='wellness-title'>Your Reflection Space</h1>", unsafe_allow_html=True)
        user_name = payload.get("username", "User")
        user_email = payload.get("email", "")

        st.success(f"Successfully authenticated as **{user_name}** ({user_email})")

        st.markdown("### Share What's Going On")
        st.write("Upload a csv, text, or Word document about what's on your mind, and you'll get a supportive reflection along with a few activities that might help.")

        # File Uploader Container
        st.markdown("<div class='wellness-card'>", unsafe_allow_html=True)
        uploaded_file = st.file_uploader("Choose a file", type=["csv", "txt", "docx"])
        st.caption("Any csv, txt, or Word (.docx) file works. There's no set format, just share whatever feels right.")
        st.markdown("</div>", unsafe_allow_html=True)

        if uploaded_file is not None:
            try:
                document_text = extract_text_from_upload(uploaded_file)

                if not document_text.strip():
                    st.error("That file appears to be empty. Please try uploading a different one.")
                else:
                    with st.spinner("Reading through what you shared..."):
                        result = get_emotional_support(document_text)

                    st.markdown("### A Reflection For You")
                    st.markdown(
                        f"<div class='wellness-card'>{result['support_message']}</div>",
                        unsafe_allow_html=True,
                    )

                    st.markdown("### A Few Things That Might Help")
                    for activity in result["activities"]:
                        st.markdown(
                            f"<div class='wellness-metric-card' style='text-align:left; margin-bottom:12px;'>{activity}</div>",
                            unsafe_allow_html=True,
                        )

            except Exception as e:
                st.error(f"Error reading file: {e}")
        else:
            st.info("Upload a file above to receive your reflection.")

        # Log Out Button
        st.write("")
        if st.button("Log Out", type="secondary", use_container_width=True):
            st.session_state.jwt_token = None
            goto("home")

        st.stop()
    else:
        st.session_state.jwt_token = None
        st.warning("Your session has expired. Please log in again.")
        goto("login")

# --- PUBLIC PAGES ---

if st.session_state.page == "home":
    # --- HOME PAGE ---
    st.markdown("<h1 class='wellness-title'>EMPLOYEE WELLNESS MANAGEMENT</h1>", unsafe_allow_html=True)
    st.subheader("Share what's on your mind, in your own words")

    st.markdown("""
    Welcome. This is a quiet space to upload what you're carrying and receive something supportive back:

    - **Secure Account Access** backed by industry-standard JWT.
    - **Upload Whatever Fits** — a csv, a text file, or a Word document, whatever you have.
    - **A Personal Reflection** along with a few activities suited to what you shared.
    """)

    st.write("")
    col1, col2 = st.columns(2)
    with col1:
        if st.button("Log In", use_container_width=True, type="primary"):
            goto("login")
    with col2:
        if st.button("Sign Up", use_container_width=True):
            goto("signup")

elif st.session_state.page == "login":
    # --- LOGIN PAGE ---
    st.markdown("<h1 class='wellness-title'>Secure Portal</h1>", unsafe_allow_html=True)
    st.subheader("Log in to your account")

    with st.form("login_form"):
        email = st.text_input("Registered Email Address")
        password = st.text_input("Password", type="password")
        submitted = st.form_submit_button("Log In", use_container_width=True)

    if submitted:
        if not email or not password:
            st.error("Please fill in both fields.")
        else:
            user = get_user_by_email(email.strip().lower())
            if not user or not verify_password(password, user["password_hash"]):
                st.error("Invalid email or password.")
            elif not user["is_verified"]:
                # Fallback in case user is not verified: redirect to verification page
                st.session_state.pending_email = user["email"]
                st.session_state.pending_username = user["username"]
                st.warning("Your account is not verified yet. Redirecting to verification page...")
                time.sleep(1.5)
                goto("verify_signup")
            else:
                token = create_jwt(user["id"], user["username"], user["email"])
                st.session_state.jwt_token = token
                st.success("Successfully logged in!")
                st.rerun()

    st.write("")
    col1, col2 = st.columns(2)
    with col1:
        if st.button("Create account", use_container_width=True):
            goto("signup")
    with col2:
        if st.button("Forgot password?", use_container_width=True):
            goto("forgot")

    if st.button("Back to Home", use_container_width=True):
        goto("home")

elif st.session_state.page == "signup":
    # --- SIGN UP PAGE ---
    st.markdown("<h1 class='wellness-title'>Join the Portal</h1>", unsafe_allow_html=True)
    st.subheader("Register a new account")

    with st.form("signup_form"):
        name = st.text_input("Name (Full Name)")
        email = st.text_input("Email Address")
        password = st.text_input("Password (min. 8 characters, letters & numbers)")
        confirm = st.text_input("Confirm Password", type="password")
        submitted = st.form_submit_button("Sign Up", use_container_width=True)

    if submitted:
        errors = []
        if not name or len(name.strip()) < 2:
            errors.append("Please enter your name (minimum 2 characters).")

        try:
            valid = validate_email(email, check_deliverability=False)
            email_normalized = valid.normalized.lower()
        except EmailNotValidError as e:
            errors.append(f"Invalid email structure: {e}")

        ok_pw, pw_msg = is_valid_password(password)
        if not ok_pw:
            errors.append(pw_msg)
        if password != confirm:
            errors.append("Passwords do not match.")

        if not errors:
            username_candidate = email_normalized.split("@")[0]
            if get_user_by_email(email_normalized):
                errors.append("An account with this email address already exists.")
            elif get_user_by_username(username_candidate):
                errors.append(f"Username '{username_candidate}' is already taken.")

        if errors:
            for err in errors:
                st.error(err)
        else:
            # Create user in database (verified by default)
            create_user(name.strip(), username_candidate, email_normalized, password)
            st.success("Account created successfully! Redirecting to login...")
            time.sleep(1.5)
            goto("login")

    st.write("")
    if st.button("Back to login", use_container_width=True):
        goto("login")

elif st.session_state.page == "verify_signup":
    # --- VERIFY OTP PAGE ---
    st.markdown("<h1 class='wellness-title'>Verify Account</h1>", unsafe_allow_html=True)
    st.subheader("Email Verification")

    email = st.session_state.pending_email
    if not email:
        st.info("No verification is currently pending. Please register first.")
        if st.button("Go to Sign Up", use_container_width=True):
            goto("signup")
    else:
        st.write(f"We've sent a 6-digit OTP to **{email}**. Please enter it below to activate your account.")

        # Display OTP helper block for easy local testing in Colab if no SMTP app password is configured
        smtp_configured = os.getenv("SMTP_EMAIL") and not os.getenv("SMTP_EMAIL").startswith("your")
        if not smtp_configured:
            debug_otp = get_latest_otp(email, "signup")
            if debug_otp:
                st.info(f"**Demo Mode**: Since SMTP is not configured, enter OTP: **{debug_otp}**")

        with st.form("verify_form"):
            code_val = st.text_input("6-digit Verification Code", max_chars=6)
            verify_submitted = st.form_submit_button("Verify Code", use_container_width=True)

        if verify_submitted:
            if verify_otp(email, code_val.strip(), "signup"):
                mark_user_verified(email)
                st.success("Account verified successfully! You can now log in.")
                st.session_state.pending_email = None
                time.sleep(2)
                goto("login")
            else:
                st.error("Invalid or expired verification code. Please try again.")

        col1, col2 = st.columns(2)
        with col1:
            if st.button("Resend Code", use_container_width=True):
                new_code = generate_otp()
                store_otp(email, new_code, "signup")
                ok, msg = send_otp_email(email, new_code, "signup")
                if ok:
                    st.success("A fresh verification code was sent to your email.")
                else:
                    st.error(msg)
        with col2:
            if st.button("Back to login", use_container_width=True):
                goto("login")

elif st.session_state.page == "forgot":
    # --- FORGOT PASSWORD PAGE ---
    st.markdown("<h1 class='wellness-title'>Forgot Password</h1>", unsafe_allow_html=True)
    st.subheader("Request Password Reset")

    with st.form("forgot_form"):
        email = st.text_input("Enter your account Email Address")
        submitted = st.form_submit_button("Send Reset Code", use_container_width=True)

    if submitted:
        email_clean = email.strip().lower()
        user = get_user_by_email(email_clean)
        if not user:
            # Secure message: Do not reveal whether user email exists
            st.info("If that email address is registered, a password reset code has been sent.")
        else:
            code = generate_otp()
            store_otp(email_clean, code, "password_reset")
            ok, msg = send_otp_email(email_clean, code, "password_reset")
            if ok:
                st.session_state.pending_email = email_clean
                st.success("Password reset code sent. Check your inbox.")
                time.sleep(1.5)
                goto("reset")
            else:
                st.error(msg)

    st.write("")
    if st.button("Back to login", use_container_width=True):
        goto("login")

elif st.session_state.page == "reset":
    # --- RESET PASSWORD PAGE ---
    st.markdown("<h1 class='wellness-title'>Reset Password</h1>", unsafe_allow_html=True)
    st.subheader("Update Credentials")

    email = st.session_state.pending_email
    if not email:
        st.info("No pending reset requests are registered.")
        if st.button("Request Reset Code", use_container_width=True):
            goto("forgot")
    else:
        st.write(f"Verify OTP sent to **{email}** and establish your new credentials.")

        # Display OTP helper block for easy local testing in Colab if no SMTP app password is configured
        smtp_configured = os.getenv("SMTP_EMAIL") and not os.getenv("SMTP_EMAIL").startswith("your")
        if not smtp_configured:
            debug_otp = get_latest_otp(email, "password_reset")
            if debug_otp:
                st.info(f"**Demo Mode**: Since SMTP is not configured, enter OTP: **{debug_otp}**")

        with st.form("reset_form"):
            otp_val = st.text_input("6-digit Reset Code", max_chars=6)
            new_pw = st.text_input("New Password", type="password")
            confirm_pw = st.text_input("Confirm New Password", type="password")
            submitted = st.form_submit_button("Reset Password", use_container_width=True)

        if submitted:
            ok_pw, pw_msg = is_valid_password(new_pw)
            if new_pw != confirm_pw:
                st.error("Passwords do not match.")
            elif not ok_pw:
                st.error(pw_msg)
            elif not verify_otp(email, otp_val.strip(), "password_reset"):
                st.error("Invalid or expired code.")
            else:
                update_user_password(email, new_pw)
                st.success("Password updated successfully! Redirecting to login portal...")
                st.session_state.pending_email = None
                time.sleep(2)
                goto("login")

        if st.button("Back to login", use_container_width=True):
            goto("login")

Overwriting app.py


In [25]:
from db import init_db
init_db()
print("✅ Connected to PostgreSQL and ensured tables exist.")

Database initialized (tables ensured).
✅ Connected to PostgreSQL and ensured tables exist.


In [26]:
from pyngrok import ngrok, conf
import subprocess, time

conf.get_default().auth_token = values["NGROK_AUTHTOKEN"]

# Kill any previous tunnels/streamlit instances from earlier runs in this session
ngrok.kill()
get_ipython().system_raw('pkill -f streamlit || true')
time.sleep(1)

# Launch Streamlit in the background, quietly, on port 8501
get_ipython().system_raw(
    'streamlit run app.py --server.port 8501 --server.headless true '
    '--server.enableCORS false --server.enableXsrfProtection false &'
)
time.sleep(4)  # give the server a moment to boot

public_url = ngrok.connect(8501, "http")
print(f"🚀 Your app is live at: {public_url}")
print("Open that URL in your browser. Leave this Colab cell/runtime running to keep it up.")

🚀 Your app is live at: NgrokTunnel: "https://twisting-possibly-magnify.ngrok-free.dev" -> "http://localhost:8501"
Open that URL in your browser. Leave this Colab cell/runtime running to keep it up.


In [27]:
from pyngrok import ngrok
ngrok.kill()
get_ipython().system_raw('pkill -f streamlit || true')
print("Stopped Streamlit and closed ngrok tunnel.")

Stopped Streamlit and closed ngrok tunnel.
